In [1]:
import numpy as np, pandas as pd, optuna, torch
from sklearn.model_selection import StratifiedKFold
from NN_model import ImprovedNN  # your model
from pathlib import Path
import json, torch

# 0) Load ALREADY-SCALED low-MW data (same feature order as baseline)
df_low = pd.read_csv("artifacts/final_train_low_MP_scaled_clustered.csv")    # <- already scaled


TARGET_COL = "MP"

exclude = {"SMILES", TARGET_COL, "MW", "MP_category_default", "Structure_Cluster"}
num_cols = df_low.select_dtypes(include=[np.number]).columns
feature_cols = [c for c in num_cols if c not in exclude]

X = df_low[feature_cols].to_numpy(np.float32)  # <-- already scaled
y = df_low[TARGET_COL].to_numpy(np.float32)
y_strat = df_low["Structure_Cluster"].astype(str).to_numpy()

# 1) Fix folds once

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
folds = [(tr, va) for tr, va in skf.split(X, y_strat)]


/opt/anaconda3/envs/Surabie_S_clean_v2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Plot the loss curve 

import matplotlib.pyplot as plt

def plot_training_progress(train_losses, val_losses, early_stop_epoch=None, title="Training and Validation Loss"):
    #train_losses / val_losses: lists of per-epoch average loss values.
    #early_stop_epoch: integer epoch number (1-based) where early stopping triggered (optional).

    epochs = range(1, len(train_losses) + 1) 
    
    plt.figure(figsize=(8, 4))
    plt.plot(epochs, train_losses, label="Training Loss")
    plt.plot(epochs, val_losses,   label="Validation Loss")

    if early_stop_epoch is not None:
        plt.axvline(x=early_stop_epoch, color='r', linestyle='--', label="Early Stop")
    else:
    # draw line at last epoch
        plt.axvline(x=len(train_losses), color='gray', linestyle='--', label="End Epoch")
    
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [3]:
#Source: https://stackoverflow.com/questions/71998978/early-stopping-in-pytorch

# Early Stopping Based on Validation Loss
class EarlyStopper:

    # If the val loss has not been improved (i.e. stayed the same or got worse) for 20 epochs in a row, the training of the model is stopped.
    def __init__(self, patience=30, min_delta=0):
        self.patience = int(patience)
        self.min_delta = float(min_delta)
        self.counter = 0
        self.best_loss = float('inf')
        self.stop = False
        self.stop_epoch = None  # remember which epoch we stopped on (for plotting)

    def early_stop(self, val_loss, epoch=None):

        #For each epoch, checks if the validation loss has improved, we reset the counter.
        # We increase the counter if there is no improvement. Once the counter reaches the patience, we stop and remember the epoch.

        # Improvement means the loss decreased by more than min_delta
        if (self.best_loss - val_loss) > self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            # No meaningful improvement this epoch
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
                self.stop_epoch = epoch
        return self.stop
    
import numpy as np
from pathlib import Path
import torch

# sklearn metrics
from sklearn.metrics import r2_score, root_mean_squared_error


def save_checkpoint(model, optimizer, epoch, train_loss, val_loss, y_train, y_val, val_loader, 
                   fold_idx, checkpoint_dir, checkpoint_tracking, is_final=False):
        
        
        # Calculate val predictions
        model.eval()
        all_preds = []
        with torch.no_grad():
            for xb, _ in val_loader:
                preds = model(xb).cpu().numpy()
                all_preds.append(preds)
        preds_val = np.concatenate(all_preds)
        
        # Calculate performance metrics from val predictions
        checkpoint_rmse = root_mean_squared_error(y_val, preds_val)
        checkpoint_r2 = r2_score(y_val, preds_val)
        checkpoint_q2 = 1 - np.sum((y_val - preds_val)**2) / np.sum((y_val - y_train.mean())**2)
        
        # Create checkpoint filename
        if is_final:
            checkpoint_filename = f"checkpoint_epoch_{epoch:03d}_final.pt"
        else:
            checkpoint_filename = f"checkpoint_epoch_{epoch:03d}.pt"
        
        checkpoint_path_full = Path(checkpoint_dir) / checkpoint_filename
        
        # Save the checkpoint
        checkpoint_data = {'epoch': epoch, 'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss, 'val_loss': val_loss, 'rmse': checkpoint_rmse, 'r2': checkpoint_r2, 'q2': checkpoint_q2,
            'fold_idx': fold_idx, 'is_final': is_final}
        torch.save(checkpoint_data, checkpoint_path_full)
        
        # Record info for spreadsheet
        checkpoint_info = {'Fold': fold_idx, 'Epoch': epoch, 'Checkpoint_Filename': checkpoint_filename, 'Checkpoint_Path': str(checkpoint_path_full),
            'Train_Loss': round(train_loss, 6), 'Val_Loss': round(val_loss, 6), 'RMSE': round(checkpoint_rmse, 6), 'R2': round(checkpoint_r2, 6), 
            'Q2': round(checkpoint_q2, 6), 'Is_Final': is_final}
        checkpoint_tracking.append(checkpoint_info)
        
        checkpoint_type = "Final" if is_final else "Regular"
        print(f"[Fold {fold_idx}] {checkpoint_type} checkpoint saved at epoch {epoch} - RMSE: {checkpoint_rmse:.4f}")
        return True

import torch
import torch.nn as nn

# since RMSE Loss is not in PyTorch, we define it here using MSELoss

class RMSELoss(nn.Module):
    def __init__(self, eps=1e-8):  

        super().__init__()
        self.mse = nn.MSELoss()
        self.eps = eps      # eps: a small constant to avoid sqrt(0) or division by zero;  to prevent potential numerical instability or "division by zero" like issues if the MSE happens to be exactly zero 

    def forward(self, y_pred, y_true):
        mse = self.mse(y_pred, y_true)
        rmse = torch.sqrt(mse + self.eps)
        return rmse

        
# ==== Standard libraries ====
import copy
from pathlib import Path

# ==== Numerical ====
import numpy as np

# ==== PyTorch ====
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# ==== sklearn metrics ====
from sklearn.metrics import r2_score, root_mean_squared_error

# ==== Your custom modules ====
from NN_model import ImprovedNN   # make sure NN_model/ folder is in sys.path
import copy 

def evaluate_fold(trial, fold_idx, X_train_scaled, y_train, X_val_scaled, y_val, hidden_layers, learning_rate, batch_size, dropout_rate, weight_decay, max_epochs = 10**9, patience = 30, min_delta = 0, X_test_scaled=None, y_test=None, save_checkpoints=True, checkpoint_dir="checkpoints", save_every_n_epochs=15):

    # Set device to CPU
    device = torch.device("cpu")
    print(f"Fold {fold_idx}: Training on cpu")

    #Setup checkpoint directory and tracking list
    checkpoint_tracking = []  # Empty list to track performance metrics for model checkpointing
    
    # If saving checkpoints is true, we are creating the path checkpoints/fold_{fold_idx}
    if save_checkpoints:
        checkpoint_path = Path(checkpoint_dir)
        checkpoint_path.mkdir(parents=True, exist_ok=True)
        fold_checkpoint_dir = checkpoint_path / f"fold_{fold_idx}"
        fold_checkpoint_dir.mkdir(parents=True, exist_ok=True)
        print(f"Checkpoints will be saved to: {fold_checkpoint_dir}")

    # Convert data to tensors and move to device
    X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32).to(device)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32).to(device)      # reshape the targets to column vectors to match the model’s predictions and prevent PyTorch from doing sneaky broadcasting
    X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32).to(device)
    y_val_tensor   = torch.tensor(y_val,   dtype=torch.float32).to(device)


    # Load the training df
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    
    #Load the val df 
    val_dataset = TensorDataset(X_val_tensor, y_val_tensor)   
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

    #model, optimizer  scheduler, loss (set up training components)
    model     = ImprovedNN(input_size = X_train_scaled.shape[1], hidden_layers=hidden_layers, dropout_rate = dropout_rate).to(device) #A new model is created for each trial run with Optuna, the hyperparameters in each trial is chosen by Optuna, new instance of the model is created, and input size is determined by features in scaled training data, drop out rate is suggested by Optuna
    criterion = RMSELoss() # changed from HuberLoss to RMSELoss 
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay) #Optimizer adjusts the model's internal weights and biases, AdamW is an optimizer, model.parameters() tells optimizer what to optimize, lr = learning_rate uses suggested learning rate by Optuna, same for weight_decay                     
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10) #Automatically adjusts learning rate during training, mode = "min" monitors metric to minimize, factor = 0.5 if monitored metric doesn't improve for a certain amount of epochs reduce lr by 1/2, patience is number of epochs to wait before adjustment by factor 
                                               
    # Set up values for early stopping
    early_stopper = EarlyStopper(patience=patience, min_delta=min_delta)

    best_val_loss = float('inf')
    best_state = copy.deepcopy(model.state_dict())

    train_losses, val_losses = [], []
    stop_epoch = None

        #-- Model Training ---
    for epoch in range(1, max_epochs + 1): ##for loop represemts the training process for a single model for the current trial, runs for 300 epochs, each epoch indicates that the model has run once, so 12 epoches means the model has been run 12 times 
        model.train()
        train_loss = 0.0
        for xb, yb in train_loader: ##Put model into training mode (dropout turn on), loops over each batch (xb = input, yb = target)
            optimizer.zero_grad() #Clear out any old gradients (a gradient is a piece of information about how much much to change the weights)
            preds = model(xb) #make predictions
            loss  = criterion(preds, yb) #Calculate loss function
            loss.backward() #Back propogate
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5) #Prevents exploding gradients which causes the model to become unstable, limits how big the adjustments to weights can be 
            optimizer.step() #Uses calculated gradients to actually update model's weights and biases trying to reduce loss 
            train_loss += loss.item()
        train_loss /= len(train_loader)
        train_losses.append(train_loss)
        
        # --- To validate the model ----
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                preds = model(xb)
                loss  = criterion(preds, yb)
                val_loss += loss.item()
        val_loss /= len(val_loader)
        val_losses.append(val_loss)   
       
        scheduler.step(val_loss)
        
        # Update best model if validation loss improves
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
        
        # Saves checkpoints every n epochs (and at epoch 1)
        if save_checkpoints and (epoch % save_every_n_epochs == 0 or epoch == 1):
            # Calculate metrics for this checkpoint
            save_checkpoint(model, optimizer, epoch, train_loss, val_loss, y_train, y_val, val_loader, fold_idx, fold_checkpoint_dir, checkpoint_tracking, is_final=False)

        # Check for early stopping
        should_stop = early_stopper.early_stop(val_loss, epoch=epoch)
        if should_stop:
            stop_epoch = early_stopper.stop_epoch
            print(f"[Fold {fold_idx}] Early stopping  at epoch {stop_epoch} (best Val Loss: {best_val_loss:.4f})")

            # Save final checkpoint on early stop (guarantee last snapshot)
            if save_checkpoints and epoch % save_every_n_epochs != 0 and epoch != 1:
                save_checkpoint(model, optimizer, epoch, train_loss, val_loss, y_train, y_val, 
                              val_loader, fold_idx, fold_checkpoint_dir, checkpoint_tracking, is_final=True)
            
            break

        # Log progress every 50 epochs or first epoch
        if epoch % 50 == 0 or epoch == 1:
            print(f"[Fold {fold_idx}] Epoch {epoch:4d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | ES {early_stopper.counter}/{patience}")
    
    # Load best model state (from epoch with lowest val loss)
    model.load_state_dict(best_state)
    model.eval()  

    # Save the checkpoint tracking spreadsheet for this fold
    if save_checkpoints and checkpoint_tracking:
        df_checkpoints = pd.DataFrame(checkpoint_tracking)
        spreadsheet_file = fold_checkpoint_dir / f"fold_{fold_idx}_checkpoints_low_test.csv"
        df_checkpoints.to_csv(spreadsheet_file, index=False)
        print(f"[Fold {fold_idx}] Checkpoint spreadsheet saved: {spreadsheet_file}")
        print(f"[Fold {fold_idx}] Total checkpoints saved: {len(checkpoint_tracking)}")
  

    # Final metrics calculation (using the best model)
    all_preds = []
    with torch.no_grad():
        for xb, _ in val_loader:
            preds = model(xb).cpu().numpy()
            all_preds.append(preds)
    preds_val = np.concatenate(all_preds)
    
    rmse = root_mean_squared_error(y_val, preds_val)
    r2 = r2_score(y_val, preds_val)
    q2 = 1 - np.sum((y_val - preds_val)**2) / np.sum((y_val - y_train.mean())**2)
 
    return rmse, r2, q2, model, train_losses, val_losses, stop_epoch


import time
import numpy as np
import torch

# Step 3: Hyperparameter optimization
trial_times = []

def objective(trial):
    # Suggest hyperparameters
    dropout_rate  = trial.suggest_float("dropout_rate",  0.2, 0.5)
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True)
    weight_decay  = trial.suggest_float("weight_decay",  1e-6, 1e-2, log=True)
    batch_size    = trial.suggest_categorical("batch_size", [16, 32, 64])

    # First hidden layer max 256
    h1 = trial.suggest_categorical("h1", [64, 96, 128, 160, 192, 224, 256])
    h2 = max(h1 // 2, 4)
    h3 = max(h2 // 2, 2)
    hidden_layers = [h1, h2, h3]

    start = time.perf_counter()

    rmses = []

    # Use folds you defined earlier
    for fold_idx, (tr_idx, val_idx) in enumerate(folds):
        X_train_scaled = X[tr_idx]
        y_train        = y[tr_idx]
        X_val_scaled   = X[val_idx]
        y_val          = y[val_idx]
        
        trial_checkpoint_root = Path("checkpoints_low_MP") / f"trial_{trial.number:03d}"

        rmse, _, _, _, _, _, _ = evaluate_fold(
            trial=trial,
            fold_idx=fold_idx,
            X_train_scaled=X_train_scaled,
            y_train=y_train,
            X_val_scaled=X_val_scaled,
            y_val=y_val,
            learning_rate=learning_rate,
            batch_size=batch_size,
            hidden_layers=hidden_layers,
            dropout_rate=dropout_rate,
            weight_decay=weight_decay,
            save_checkpoints=True,
            checkpoint_dir=trial_checkpoint_root
        )
        rmses.append(rmse)

    elapsed = (time.perf_counter() - start) / 60.0
    trial_times.append(elapsed)
    print(f"Trial {trial.number} finished in {elapsed:.2f} minutes")

    avg_rmse = float(np.mean(rmses))
    print(f"Trial {trial.number}: Average RMSE = {avg_rmse:.4f}")
    return avg_rmse

In [4]:
import time 
import torch
import optuna
from optuna.importance import get_param_importances
import optuna.visualization as vis 
import copy

device = torch.device("cpu")

def set_optuna_study(n_trials): 
    start_time = time.perf_counter()
    print("Setting up Optuna study...")
    
    # 1) Set up the Optuna study
    study = optuna.create_study(direction='minimize') #minimize return loss
    study.optimize(objective, n_trials=n_trials)  #CHANGE TO 100 AFTER TESTING
    
    # 2) Identify the best hyperparameters
    best_params = study.best_params #best_params holds the dropout, learning rate, and weight decay that gave the lowest best_val_loss
    print("Best hyperparameters:", best_params)
    
    end_time = time.perf_counter()
    elapsed_time = (end_time - start_time) / 60.0
    print(f"Optuna study completed in {elapsed_time:.2f} minutes")
    
    return best_params, study

best_params, study = set_optuna_study(n_trials=20) 

[I 2025-11-30 16:20:56,172] A new study created in memory with name: no-name-7ec34582-fa53-4afb-aa6c-627898caa237


Setting up Optuna study...
Fold 0: Training on cpu
Checkpoints will be saved to: checkpoints_low_MP/trial_000/fold_0
[Fold 0] Regular checkpoint saved at epoch 1 - RMSE: 55.5236
[Fold 0] Epoch    1 | Train Loss: 53.8137 | Val Loss: 54.2923 | ES 0/30
[Fold 0] Regular checkpoint saved at epoch 15 - RMSE: 54.3081
[Fold 0] Regular checkpoint saved at epoch 30 - RMSE: 51.3840
[Fold 0] Regular checkpoint saved at epoch 45 - RMSE: 49.9764
[Fold 0] Epoch   50 | Train Loss: 46.9410 | Val Loss: 48.1370 | ES 4/30
[Fold 0] Regular checkpoint saved at epoch 60 - RMSE: 46.4235
[Fold 0] Regular checkpoint saved at epoch 75 - RMSE: 44.3690
[Fold 0] Regular checkpoint saved at epoch 90 - RMSE: 42.3783
[Fold 0] Epoch  100 | Train Loss: 39.5215 | Val Loss: 40.1130 | ES 5/30
[Fold 0] Regular checkpoint saved at epoch 105 - RMSE: 40.3261
[Fold 0] Regular checkpoint saved at epoch 120 - RMSE: 37.1351
[Fold 0] Regular checkpoint saved at epoch 135 - RMSE: 35.0855
[Fold 0] Regular checkpoint saved at epoch 15

[I 2025-11-30 16:25:18,382] Trial 0 finished with value: 28.254548645019533 and parameters: {'dropout_rate': 0.4935861210543797, 'learning_rate': 8.017892975646732e-05, 'weight_decay': 7.775575306974708e-05, 'batch_size': 32, 'h1': 64}. Best is trial 0 with value: 28.254548645019533.


[Fold 9] Regular checkpoint saved at epoch 255 - RMSE: 31.5622
[Fold 9] Early stopping  at epoch 255 (best Val Loss: 29.2261)
[Fold 9] Checkpoint spreadsheet saved: checkpoints_low_MP/trial_000/fold_9/fold_9_checkpoints_low_test.csv
[Fold 9] Total checkpoints saved: 18
Trial 0 finished in 4.37 minutes
Trial 0: Average RMSE = 28.2545
Fold 0: Training on cpu
Checkpoints will be saved to: checkpoints_low_MP/trial_001/fold_0
[Fold 0] Regular checkpoint saved at epoch 1 - RMSE: 54.2148
[Fold 0] Epoch    1 | Train Loss: 53.7107 | Val Loss: 51.5740 | ES 0/30
[Fold 0] Regular checkpoint saved at epoch 15 - RMSE: 41.9135
[Fold 0] Regular checkpoint saved at epoch 30 - RMSE: 31.4508
[Fold 0] Regular checkpoint saved at epoch 45 - RMSE: 24.9948
[Fold 0] Epoch   50 | Train Loss: 24.5607 | Val Loss: 25.4441 | ES 4/30
[Fold 0] Regular checkpoint saved at epoch 60 - RMSE: 23.4211
[Fold 0] Regular checkpoint saved at epoch 75 - RMSE: 22.6783
[Fold 0] Regular checkpoint saved at epoch 90 - RMSE: 22.235

[I 2025-11-30 16:30:05,650] Trial 1 finished with value: 24.0030797958374 and parameters: {'dropout_rate': 0.2731616907813689, 'learning_rate': 0.0002704775135570454, 'weight_decay': 6.80340410221934e-05, 'batch_size': 64, 'h1': 224}. Best is trial 1 with value: 24.0030797958374.


...

[Fold 9] Early stopping  at epoch 118 (best Val Loss: 24.8602)
[Fold 9] Final checkpoint saved at epoch 118 - RMSE: 25.3934
[Fold 9] Checkpoint spreadsheet saved: checkpoints_low_MP/trial_017/fold_9/fold_9_checkpoints_low_test.csv
[Fold 9] Total checkpoints saved: 9
Trial 17 finished in 1.83 minutes
Trial 17: Average RMSE = 24.0490
Fold 0: Training on cpu
Checkpoints will be saved to: checkpoints_low_MP/trial_018/fold_0
[Fold 0] Regular checkpoint saved at epoch 1 - RMSE: 55.5297
[Fold 0] Epoch    1 | Train Loss: 54.1538 | Val Loss: 52.7611 | ES 0/30
[Fold 0] Regular checkpoint saved at epoch 15 - RMSE: 55.3397
[Fold 0] Regular checkpoint saved at epoch 30 - RMSE: 55.1521
[Fold 0] Regular checkpoint saved at epoch 45 - RMSE: 54.9842
[Fold 0] Epoch   50 | Train Loss: 53.6184 | Val Loss: 52.2202 | ES 1/30
[Fold 0] Regular checkpoint saved at epoch 60 - RMSE: 54.8193
[Fold 0] Regular checkpoint saved at epoch 75 - RMSE: 54.6275
[Fold 0] Regular checkpoint saved at epoch 90 - RMSE: 54.5030

[I 2025-11-30 17:46:41,266] Trial 18 finished with value: 50.8254810333252 and parameters: {'dropout_rate': 0.38287166239006826, 'learning_rate': 1.016226861488772e-05, 'weight_decay': 0.002163616720493989, 'batch_size': 64, 'h1': 64}. Best is trial 5 with value: 23.87682971954346.


[Fold 9] Early stopping  at epoch 492 (best Val Loss: 49.2499)
[Fold 9] Final checkpoint saved at epoch 492 - RMSE: 50.4621
[Fold 9] Checkpoint spreadsheet saved: checkpoints_low_MP/trial_018/fold_9/fold_9_checkpoints_low_test.csv
[Fold 9] Total checkpoints saved: 34
Trial 18 finished in 4.26 minutes
Trial 18: Average RMSE = 50.8255
Fold 0: Training on cpu
Checkpoints will be saved to: checkpoints_low_MP/trial_019/fold_0
[Fold 0] Regular checkpoint saved at epoch 1 - RMSE: 53.8098
[Fold 0] Epoch    1 | Train Loss: 52.4732 | Val Loss: 52.3051 | ES 0/30
[Fold 0] Regular checkpoint saved at epoch 15 - RMSE: 33.6716
[Fold 0] Regular checkpoint saved at epoch 30 - RMSE: 27.1626
[Fold 0] Regular checkpoint saved at epoch 45 - RMSE: 24.5002
[Fold 0] Epoch   50 | Train Loss: 26.4749 | Val Loss: 23.4522 | ES 17/30
[Fold 0] Regular checkpoint saved at epoch 60 - RMSE: 23.3653
[Fold 0] Regular checkpoint saved at epoch 75 - RMSE: 23.8298
[Fold 0] Regular checkpoint saved at epoch 90 - RMSE: 23.05

[I 2025-11-30 17:52:45,073] Trial 19 finished with value: 24.368821716308595 and parameters: {'dropout_rate': 0.23758773547552203, 'learning_rate': 0.00011621932933112088, 'weight_decay': 8.726050898241297e-05, 'batch_size': 16, 'h1': 256}. Best is trial 5 with value: 23.87682971954346.


[Fold 9] Early stopping  at epoch 119 (best Val Loss: 25.2220)
[Fold 9] Final checkpoint saved at epoch 119 - RMSE: 27.3280
[Fold 9] Checkpoint spreadsheet saved: checkpoints_low_MP/trial_019/fold_9/fold_9_checkpoints_low_test.csv
[Fold 9] Total checkpoints saved: 9
Trial 19 finished in 6.06 minutes
Trial 19: Average RMSE = 24.3688
Best hyperparameters: {'dropout_rate': 0.29826065162229914, 'learning_rate': 0.0004690323248859974, 'weight_decay': 1.184181600092945e-05, 'batch_size': 32, 'h1': 256}
Optuna study completed in 91.81 minutes


In [5]:
# Save the best parameters

print("Best Trial Number:", study.best_trial.number)
print("  RMSE:", study.best_value)
print("  Params:", study.best_params)

Best Trial Number: 5
  RMSE: 23.87682971954346
  Params: {'dropout_rate': 0.29826065162229914, 'learning_rate': 0.0004690323248859974, 'weight_decay': 1.184181600092945e-05, 'batch_size': 32, 'h1': 256}


In [ ]:
from pathlib import Path
import torch
import pandas as pd

# BASE and artifacts_dir should already be defined, but just in case:
BASE = Path.cwd()  # transfer_learning
artifacts_dir = BASE / "artifacts"

# Directory to store the final best models for each fold
best_models_dir = artifacts_dir / "int_Mw_best_models"
best_models_dir.mkdir(parents=True, exist_ok=True)

# (Optional) directory to store checkpoints from this final run
final_ckpt_dir = BASE / "checkpoints_int_MP_best"
final_ckpt_dir.mkdir(parents=True, exist_ok=True)

# Make sure best_params exists (from your Optuna study)
print("Best hyperparameters from Optuna:", best_params)

# Helper to derive hidden layers from best_params (same logic as in objective)
def build_hidden_layers_from_best(best_params):
    h1 = best_params["h1"]
    h2 = max(h1 // 2, 4)
    h3 = max(h2 // 2, 2)
    return [h1, h2, h3]

hidden_layers = build_hidden_layers_from_best(best_params)
dropout_rate  = best_params["dropout_rate"]
learning_rate = best_params["learning_rate"]
weight_decay  = best_params["weight_decay"]
batch_size    = best_params["batch_size"]

print("Using hidden_layers:", hidden_layers)
print("dropout:", dropout_rate, "| lr:", learning_rate, "| wd:", weight_decay, "| batch_size:", batch_size)

# To keep track of metrics across folds
final_fold_metrics = []

# ---------- Final training loop for all 10 folds ----------
for fold in range(10):
    print(f"\n==================== Final training for fold {fold} ====================")
    
    X_train_scaled = fold_data[fold]["X_train"]
    y_train        = fold_data[fold]["y_train"]
    X_val_scaled   = fold_data[fold]["X_val"]
    y_val          = fold_data[fold]["y_val"]

    # Train this fold with the best hyperparameters
    rmse, r2, q2, model, train_losses, val_losses, stop_epoch = evaluate_fold(
        trial=None,                    # not needed for final run
        fold_idx=fold,
        X_train_scaled=X_train_scaled,
        y_train=y_train,
        X_val_scaled=X_val_scaled,
        y_val=y_val,
        hidden_layers=hidden_layers,
        learning_rate=learning_rate,
        batch_size=batch_size,
        dropout_rate=dropout_rate,
        weight_decay=weight_decay,
        max_epochs=10**9,              # will stop via early-stopping anyway
        patience=30,
        min_delta=0.0,
        save_checkpoints=True,         # save checkpoints for this BEST config
        checkpoint_dir=final_ckpt_dir,
        save_every_n_epochs=15
    )

    # ---------- Save the final model for this fold ----------
    model_path = best_models_dir / f"int_Mw_best_fold_{fold}.pt"
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "hidden_layers": hidden_layers,
            "dropout_rate": dropout_rate,
            "learning_rate": learning_rate,
            "weight_decay": weight_decay,
            "batch_size": batch_size,
            "fold_idx": fold,
            "rmse": rmse,
            "r2": r2,
            "q2": q2,
        },
        model_path,
    )
    print(f"[Fold {fold}] Saved best model to: {model_path}")

    # store metrics
    final_fold_metrics.append(
        {
            "Fold": fold,
            "RMSE": rmse,
            "R2": r2,
            "Q2": q2,
            "Stop_Epoch": stop_epoch,
            "Model_Path": str(model_path),
        }
    )

# ---------- Save a summary CSV of all folds ----------
metrics_df = pd.DataFrame(final_fold_metrics)
metrics_path = best_models_dir / "int_MP_best_models_summary.csv"
metrics_df.to_csv(metrics_path, index=False)
print("\n✅ Saved summary of best models across folds to:", metrics_path)
print(metrics_df)


Best hyperparameters from Optuna: {'dropout_rate': 0.38320113538059225, 'learning_rate': 0.0009445631572695242, 'weight_decay': 0.0006404528742319037, 'batch_size': 16, 'h1': 256}
Using hidden_layers: [256, 128, 64]
dropout: 0.38320113538059225 | lr: 0.0009445631572695242 | wd: 0.0006404528742319037 | batch_size: 16

==================== Final training for fold 0 ====================
Fold 0: Training on cpu
Checkpoints will be saved to: /Users/sdl5_mp/Documents/GitHub/SDL5_MP/transfer_learning/checkpoints_int_Mw_best/fold_0
[Fold 0] Regular checkpoint saved at epoch 1 - RMSE: 106.1394
[Fold 0] Epoch    1 | Train Loss: 126.9957 | Val Loss: 105.7333 | ES 0/30
[Fold 0] Regular checkpoint saved at epoch 15 - RMSE: 33.5823
[Fold 0] Regular checkpoint saved at epoch 30 - RMSE: 33.0599
[Fold 0] Regular checkpoint saved at epoch 45 - RMSE: 32.5848
[Fold 0] Epoch   50 | Train Loss: 35.1822 | Val Loss: 31.5489 | ES 0/30
[Fold 0] Regular checkpoint saved at epoch 60 - RMSE: 31.9089
[Fold 0] Regul

In [ ]:
# Plotting optuna trial with RMSE

import matplotlib.pyplot as plt
import pandas as pd

# Collect data from your study
records = []
for t in study.trials:
    if t.value is not None:  # skip failed/pruned trials
        records.append({"trial": t.number, "rmse": t.value})

df = pd.DataFrame(records)

plt.figure(figsize=(8,5))
plt.plot(df["trial"], df["rmse"], marker="o")
plt.title("Optuna Trials vs Mean RMSE (10-Fold Average)")
plt.xlabel("Trial Number")
plt.ylabel("Average RMSE")
plt.grid(True)
plt.show()

In [ ]:
# For Each trial, save the model for each fold?
# when saving checkpoints, is it saving the whole model?
# differenc between epochs saving and the model saving



In [ ]:
# Load the best model (best trial, best fold)


# Grab the test data and run it